In [558]:
import pandas as pd


match = pd.read_csv("./data/gold_match.csv")
match_tickets = pd.read_csv("./data/gold_match_tickets.csv")


In [559]:
match

,match_id,match_date,kickoff_time_local,match_date_utc,kickoff_time_utc,matchday,competition_id,competition_name,season,season_id,...,venue,winner,result_home,goals_home_ht,goals_away_ht,goals_home_ft,goals_away_ft,is_home_match,last_result_vs_opponent,tickets_scanned
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022-07-23Z,16:15:00Z,1.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Guldensporenstadion,away,L,0,0,0,2,False,L 1-2,NaN
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022-07-30Z,16:15:00Z,2.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,home,W,1,0,2,0,True,NaN,5565.0
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,2022-08-07Z,19:00:00Z,3.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Bosuilstadion,home,W,2,1,4,2,False,L 0-1,NaN
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,2022-08-14Z,16:30:00Z,4.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,away,L,0,2,0,3,True,L 1-4,7440.0
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,2022-08-21Z,16:30:00Z,5.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Stade Maurice Dufrasne,away,L,0,2,1,3,False,W 2-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,esy974nh0dqx61hrn6g35y044,2026-02-07,20:45:00,2026-02-07Z,19:45:00Z,24.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,Planet Group Arena,away,L,0,2,1,3,False,W 4-0,61.0
138,ewgb5fczpfa71pfzihlip6yok,2026-02-14,16:00:00,2026-02-14Z,15:00:00Z,25.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,King Power at Den Dreef Stadion,home,W,2,1,3,2,True,W 1-0,5137.0
139,exzov7o3p9v5vqr86qipfm1w4,2026-02-21,18:15:00,2026-02-21Z,17:15:00Z,26.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,Jan Breydelstadion,home,W,0,0,2,1,False,L 0-1,NaN
140,f19ucr3rev3uy5uvxjxznqsk4,2026-02-28,20:45:00,2026-02-28Z,19:45:00Z,27.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2025/2026,16lkdpoccbh056r0v0bordo2c,...,Lotto Park,home,W,1,1,5,1,False,D 1-1,NaN


In [560]:
import pandas as pd
import pandas as pd

def sold_vs_scanned(df_match_tickets, df_match):
    
    # Selecionar as colunas
    tickets_df = df_match_tickets[['match_id', 'tickets_sold_total', 'seasonpass_holders']]
    match_df = df_match[['match_id', 'tickets_scanned']]
    
    # Fazer o merge
    df_sold_scanned = pd.merge(tickets_df, match_df, on='match_id', how='inner')
    
    # Excluir nulos (substituir por 0)
    df_sold_scanned['tickets_sold_total'] = df_sold_scanned['tickets_sold_total'].fillna(0)
    df_sold_scanned['seasonpass_holders'] = df_sold_scanned['seasonpass_holders'].fillna(0)
    df_sold_scanned['tickets_scanned'] = df_sold_scanned['tickets_scanned'].fillna(0)
    
    # --- NOVA LÓGICA AQUI ---
    # Como os seasonpass já estão incluídos, o total esperado é apenas o total de bilhetes vendidos
    df_sold_scanned['total_expected'] = df_sold_scanned['tickets_sold_total']

    # Calcular No-shows (absoluto)
    df_sold_scanned['no_shows'] = df_sold_scanned['total_expected'] - df_sold_scanned['tickets_scanned']
    
    # Calcular a Taxa de No-shows (percentagem) para teres correlações mais precisas
    # (Evitar divisão por zero caso haja algum jogo com 0 bilhetes vendidos)
    df_sold_scanned['no_show_rate'] = df_sold_scanned.apply(
        lambda row: row['no_shows'] / row['total_expected'] if row['total_expected'] > 0 else 0, 
        axis=1
    )
    
    return df_sold_scanned

# Atualizar o dataframe com a função corrigida
df_resultado = sold_vs_scanned(match_tickets, match)

df_resultado[['match_id', 'total_expected', 'tickets_scanned', 'no_shows', 'no_show_rate']]






,match_id,total_expected,tickets_scanned,no_shows,no_show_rate
0,3kewmr690bhoid47mku9wfgno,5988,7786.0,-1798.0,-0.300267
1,3nrkjdhpc5zse1sdzt1onor2s,6652,8577.0,-1925.0,-0.289387
2,3rwouz4640n6ngfbwrwsqkp3o,7385,9144.0,-1759.0,-0.238186
3,3vk3btgbt7w7qy9hjp3f2934k,6929,9598.0,-2669.0,-0.385193
4,3yp7glzujzj8nc07qp0sd0kr8,6901,9211.0,-2310.0,-0.334734
...,...,...,...,...,...
66,ecz9iyxyc5ira9n59m8cp3bis,7473,5812.0,1661.0,0.222267
67,enw1n6mhkzvb6uptwhzs4gowk,6720,5322.0,1398.0,0.208036
68,eqncupz92qy89z9bxks1e0t90,7355,5971.0,1384.0,0.188171
69,ewgb5fczpfa71pfzihlip6yok,7118,5137.0,1981.0,0.278309


In [561]:
# correlation betewen no shows and weather

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

match_context = pd.read_csv("./data/gold_match_context.csv")

df_match = pd.read_csv("./data/gold_match.csv")


df_clima = match_context[['match_id', 'weather_temp_mean_c', 'weather_precipitation_mm', 'weather_snowfall_cm']]

df_final = pd.merge(df_resultado, df_clima, on='match_id', how='inner')
df_final = pd.merge(df_final, df_match, on='match_id', how='inner')

tabela_correlacao = df_final[['no_shows','weather_temp_mean_c', 'weather_precipitation_mm']].corr()

tabela_correlacao




,no_shows,weather_temp_mean_c,weather_precipitation_mm
no_shows,1.000000,-0.069410,-0.008404
weather_temp_mean_c,-0.069410,1.000000,0.241213
weather_precipitation_mm,-0.008404,0.241213,1.000000


In [562]:
import pandas as pd
import seaborn as sns

match_context = pd.read_csv("./data/gold_match_context.csv")

df_calendario = match_context[['match_id', 'match_date', 'weekday', 'weekday_name', 'is_weekend']]

df_final_dias = pd.merge(df_resultado, df_calendario, on='match_id', how='inner')




media_por_dia = df_final_dias.groupby('weekday_name')['no_shows'].mean().reset_index()

# order week days
ordem_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
media_por_dia['weekday_name'] = pd.Categorical(media_por_dia['weekday_name'], categories=ordem_dias, ordered=True)
media_por_dia = media_por_dia.sort_values('weekday_name')

media_por_dia






,weekday_name,no_shows
3,Tuesday,353.00
4,Wednesday,-615.00
0,Friday,267.70
1,Saturday,-102.00
2,Sunday,-177.25


In [563]:
# < 1000
jogos_suspeitos = df_resultado[df_resultado['tickets_scanned'] < 1000]

# 2. Escolhemos as colunas mais importantes para perceber o que aconteceu
colunas_para_ver = ['tickets_sold_total', 'seasonpass_holders', 'total_expected', 'tickets_scanned', 'no_shows']

# 3. Mostramos os resultados
print(f"Found {len(jogos_suspeitos)} games with less than 1000 people")
jogos_suspeitos[colunas_para_ver]

Found 0 games with less than 1000 people


,tickets_sold_total,seasonpass_holders,total_expected,tickets_scanned,no_shows


In [564]:
# order by most no shows
top_no_shows = df_final_dias.sort_values(by='no_shows', ascending=False)

colunas_para_ver = ['match_id', 'tickets_sold_total', 'seasonpass_holders', 'total_expected', 'tickets_scanned', 'no_shows']


top_no_shows[colunas_para_ver]

,match_id,tickets_sold_total,seasonpass_holders,total_expected,tickets_scanned,no_shows
37,9bwhkknuamvj2f60dtr3tdlas,6708,4405,6708,4350.0,2358.0
69,ewgb5fczpfa71pfzihlip6yok,7118,4235,7118,5137.0,1981.0
38,9garbt9ourfmmy4uxcq3f84yc,6878,4405,6878,4913.0,1965.0
28,7wei2vfmf9wa2ikr3hmt50ljo,7743,4546,7743,5782.0,1961.0
65,e6vo7nt78kq3sax8h8o1ke4gk,6278,4235,6278,4494.0,1784.0
...,...,...,...,...,...,...
14,57i6yyarvjpugxfcvu1v2fl04,8194,4405,8194,11069.0,-2875.0
62,dyrd81lg1d8eb5ptsbq1tg6j8,4937,4032,4937,8310.0,-3373.0
58,dt091ishr3km0wuuhiatev9xw,5721,4032,5721,9331.0,-3610.0
63,e0tuth9cc11wefswrzq1cjyms,5962,4032,5962,10079.0,-4117.0


In [565]:
# order by most no shows and weekdays
top_no_shows = df_final_dias.sort_values(by='no_shows', ascending=False)

colunas_para_ver = [
    'match_id', 
    'match_date', 
    'weekday_name', 
    'tickets_sold_total', 
    'seasonpass_holders', 
    'total_expected', 
    'tickets_scanned', 
    'no_shows',  
]

top_no_shows[colunas_para_ver]

,match_id,match_date,weekday_name,tickets_sold_total,seasonpass_holders,total_expected,tickets_scanned,no_shows
37,9bwhkknuamvj2f60dtr3tdlas,2024-04-20,Saturday,6708,4405,6708,4350.0,2358.0
69,ewgb5fczpfa71pfzihlip6yok,2026-02-14,Saturday,7118,4235,7118,5137.0,1981.0
38,9garbt9ourfmmy4uxcq3f84yc,2024-05-05,Sunday,6878,4405,6878,4913.0,1965.0
28,7wei2vfmf9wa2ikr3hmt50ljo,2025-02-15,Saturday,7743,4546,7743,5782.0,1961.0
65,e6vo7nt78kq3sax8h8o1ke4gk,2025-12-07,Sunday,6278,4235,6278,4494.0,1784.0
...,...,...,...,...,...,...,...,...
14,57i6yyarvjpugxfcvu1v2fl04,2024-03-17,Sunday,8194,4405,8194,11069.0,-2875.0
62,dyrd81lg1d8eb5ptsbq1tg6j8,2023-04-08,Saturday,4937,4032,4937,8310.0,-3373.0
58,dt091ishr3km0wuuhiatev9xw,2023-02-26,Sunday,5721,4032,5721,9331.0,-3610.0
63,e0tuth9cc11wefswrzq1cjyms,2023-04-23,Sunday,5962,4032,5962,10079.0,-4117.0


In [566]:
import pandas as pd

# Carregar os dados dos bilhetes
match_tickets = pd.read_csv("./data/gold_match_tickets.csv")


# Escolher apenas as colunas que queres inspecionar
colunas_simples = [
    'match_id',
    'tickets_sold_total',
    'tickets_trib1',
    'tickets_trib2_thuis',
    'tickets_trib2_uit',
    'tickets_trib3',
    'tickets_trib4'
]

# Criar a tabela simples e mostrar as primeiras 10 a 15 linhas
tabela_simples = match_tickets[colunas_simples]

# Mostrar a tabela
tabela_simples.head(15)

,match_id,tickets_sold_total,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4
0,3kewmr690bhoid47mku9wfgno,5988,2088,427,1,1903,1569
1,3nrkjdhpc5zse1sdzt1onor2s,6652,2256,517,1,2307,1571
2,3rwouz4640n6ngfbwrwsqkp3o,7385,2405,572,17,2821,1570
3,3vk3btgbt7w7qy9hjp3f2934k,6929,2441,584,0,2334,1570
4,3yp7glzujzj8nc07qp0sd0kr8,6901,2329,566,0,2437,1569
5,42wgdmxzlcn9w08umkea3hpg4,7617,2852,584,198,2412,1571
6,4729fykixfyfwxlbno9zilq8k,7955,2714,588,170,2913,1570
7,4bjgrdur7wdidfbs58xmjynf8,8132,2841,593,0,3126,1572
8,4i4bz7rnjs44z8bt8bnftavbo,7181,2525,588,0,2495,1573
9,4mbxzkhlc9edsor1s0fjsxses,7217,2437,587,23,2599,1571


In [567]:
import pandas as pd

# Carregar os dados
match_tickets = pd.read_csv("./data/gold_match_tickets.csv")

# Definir as colunas das bancadas
colunas_tribunas = [
    'tickets_trib1',
    'tickets_trib2_thuis',
    'tickets_trib2_uit',
    'tickets_trib3',
    'tickets_trib4'
]



match_tickets[colunas_tribunas] = match_tickets[colunas_tribunas].fillna(0)
match_tickets['tickets_sold_total'] = match_tickets['tickets_sold_total'].fillna(0)


match_tickets['sum_tribunas'] = match_tickets[colunas_tribunas].sum(axis=1)




# Escolher as colunas para visualizar
colunas_finais = [
    'match_id', 
    'tickets_sold_total', 
    'sum_tribunas', 
    'tickets_sold_b2c',
    'tickets_sold_b2b'
]

# Mostrar as primeiras 15 linhas
match_tickets[colunas_finais].head(15)


# important

# tickets_sold_total is the sum of trib

,match_id,tickets_sold_total,sum_tribunas,tickets_sold_b2c,tickets_sold_b2b
0,3kewmr690bhoid47mku9wfgno,5988,5988,1095,882
1,3nrkjdhpc5zse1sdzt1onor2s,6652,6652,1807,953
2,3rwouz4640n6ngfbwrwsqkp3o,7385,7385,2793,880
3,3vk3btgbt7w7qy9hjp3f2934k,6929,6929,1981,1003
4,3yp7glzujzj8nc07qp0sd0kr8,6901,6901,1936,925
5,42wgdmxzlcn9w08umkea3hpg4,7617,7617,2619,1009
6,4729fykixfyfwxlbno9zilq8k,7955,7955,2943,987
7,4bjgrdur7wdidfbs58xmjynf8,8132,8132,2963,1585
8,4i4bz7rnjs44z8bt8bnftavbo,7181,7181,2264,874
9,4mbxzkhlc9edsor1s0fjsxses,7217,7217,2231,916


In [568]:
# collumn with sum of sold_total + b2b + b2c

match = pd.read_csv("./data/gold_match.csv")

# bring the tickets scanned
match_tickets = pd.merge(match_tickets, match[['match_id', 'tickets_scanned']], on='match_id', how='left')

column_everything = [
    'tickets_sold_total',
    'tickets_sold_b2c',
    'tickets_sold_b2b'
]

# collumn with sum of sold_total + b2b + b2c
match_tickets['total_tickets_sold_sum'] = match_tickets[column_everything].sum(axis = 1)

colum_final = [
    'match_id', 
    'tickets_sold_total', 
    'total_tickets_sold_sum',
    'tickets_scanned'
]

match_tickets[colum_final].head(15)


# important

#  less tickets fo the "garbage"

,match_id,tickets_sold_total,total_tickets_sold_sum,tickets_scanned
0,3kewmr690bhoid47mku9wfgno,5988,7965,7786.0
1,3nrkjdhpc5zse1sdzt1onor2s,6652,9412,8577.0
2,3rwouz4640n6ngfbwrwsqkp3o,7385,11058,9144.0
3,3vk3btgbt7w7qy9hjp3f2934k,6929,9913,9598.0
4,3yp7glzujzj8nc07qp0sd0kr8,6901,9762,9211.0
5,42wgdmxzlcn9w08umkea3hpg4,7617,11245,7784.0
6,4729fykixfyfwxlbno9zilq8k,7955,11885,9240.0
7,4bjgrdur7wdidfbs58xmjynf8,8132,12680,10546.0
8,4i4bz7rnjs44z8bt8bnftavbo,7181,10319,8038.0
9,4mbxzkhlc9edsor1s0fjsxses,7217,10364,7863.0


In [ ]:
# deference between, order by most no shows, and showing the date

match_tickets = pd.merge(match_tickets, match[['match_id', 'match_date']], on='match_id', how='left')

# 2. Calcular a diferença diretamente com as colunas que já existem
match_tickets['dif_scanned_sum'] = match_tickets['tickets_scanned'] - match_tickets['total_tickets_sold_sum']


# 1. Colocar a coluna da diferença em valor absoluto (sempre positivo)
match_tickets['dif_scanned_sum'] = match_tickets['dif_scanned_sum'].abs()

# 2. Ordenar por essa coluna do mais alto para o mais baixo (descendente)
match_tickets = match_tickets.sort_values(by='dif_scanned_sum', ascending=False )



# 3. Escolher as colunas para mostrar
colunas_finais = [
    'match_id',
    'match_date',
    'total_tickets_sold_sum',
    'tickets_scanned',
    'dif_scanned_sum'
]

# Mostrar as primeiras 15 linhas
match_tickets[colunas_finais].head(15)

,match_id,match_date,total_tickets_sold_sum,tickets_scanned,dif_scanned_sum
69,ewgb5fczpfa71pfzihlip6yok,2026-02-14,11929,5137.0,6792.0
68,eqncupz92qy89z9bxks1e0t90,2026-02-01,12725,5971.0,6754.0
51,dh7ze0l15ri59fuhmnrs8pst0,2025-09-26,13473,6977.0,6496.0
55,dob21hsimmlz7agqzo8irt5w4,2025-10-18,14092,7992.0,6100.0
41,d0y4sncaf09mzv9ms24ck6olw,2025-08-15,12190,6360.0,5830.0
70,rf7k8o1xjf0xw5zrgpbtmhg,2026-03-07,11847,6062.0,5785.0
66,ecz9iyxyc5ira9n59m8cp3bis,2025-12-21,11519,5812.0,5707.0
28,7wei2vfmf9wa2ikr3hmt50ljo,2025-02-15,11396,5782.0,5614.0
23,77idfj6cmoadi2223efythatw,2024-12-14,11075,5641.0,5434.0
64,e1650lh8xyaev5kzm5v3m2kgk,2025-11-23,11049,5661.0,5388.0
